<a href="https://colab.research.google.com/github/fahmiprasetiyo/data-science-2026/blob/main/Pertemuan10_FahmiPrasetiyoHadiatna_240401020107.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 10: Algoritma Klasifikasi (Bagian 2)
* Fahmi Prasetiyo Hadiatna
* 240401020107
* IF-403

###1. Load library dan Eksplorasi data

In [8]:
# Dataset import dari Kaggle

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

# The correct file name for this dataset is 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())
print(df.shape)
print(df["Churn"].value_counts(normalize=True))

/tmp/ipykernel_813/443352655.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'telco-customer-churn' dataset.
First 5 records:    customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic            

###2. Preprocessing

In [15]:
from sklearn.model_selection import train_test_split
import numpy as np

# Convert 'TotalCharges' to numeric, handling missing values and potential non-numeric entries
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Convert 'Churn' column to numerical (0 and 1)
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# Drop 'customerID' as it's an identifier and not a feature for the model
df_processed = df.drop('customerID', axis=1)

# Identify categorical columns (object type) for one-hot encoding
categorical_cols = df_processed.select_dtypes(include='object').columns

# Apply one-hot encoding to categorical features
df_encoded = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

# Separate X (features) and y (target = Churn)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Split the data into training and testing sets
X_tr, X_te, y_tr, y_te = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42)

###3. Latih model

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

###4. Evaluasi

In [11]:
from sklearn.metrics import classification_report, roc_auc_score
# TODO: hitung prediksi dan probabilitas
# 1. Hitung prediksi label dan probabilitas untuk data uji (X_te)
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]  # Probabilitas untuk kelas positif (churn)

# TODO: tampilkan classification_report dan ROC-AUC
# 2. Tampilkan classification report
print("=== Classification Report ===")
print(classification_report(y_te, y_pred))

# 3. Tampilkan nilai ROC-AUC
roc_auc = roc_auc_score(y_te, y_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8246


###5. Prediksi probabilitas dan buat kesimpulan

In [12]:
# Menampilkan beberapa hasil probabilitas churn pelanggan sebagai contoh
print("\n=== Contoh Probabilitas Churn Pelanggan ===")
print(y_proba[:10])


=== Contoh Probabilitas Churn Pelanggan ===
[0.         0.78666667 0.09       0.28       0.         0.41666667
 0.39333333 0.11       0.00666667 0.46      ]


###Kesimpulan:

#####Model Random Forest dengan penanganan class_weight='balanced' berhasil meningkatkan nilai recall pada kelas minoritas (churn). Hal ini sangat krusial bagi perusahaan agar dapat mengidentifikasi sebanyak mungkin pelanggan yang berisiko tinggi untuk berhenti berlangganan sejak dini. Meskipun terdapat sedikit penurunan pada nilai precision akibat alarm palsu, performa keseluruhan model yang ditunjukkan oleh skor ROC-AUC tetap berada pada kategori yang baik dan stabil.